# Stress Score Prediction — Final Submission(AH_04_김지영)

- **최종 모델**: RBF-SVR
- **핵심 처리**: `mean_working` 결측군을 실제 근무시간 범위 밖으로 분리
- **입력 구성**: 범주형 변수 one-hot 정리 + 대사지표 조합 파생변수
- **후처리**: `clip(0, 1)` + `round(2)`
- **10-fold CV MAE**: 0.134170
- **최종 LB MAE**: 0.129992

중간 EDA와 실패 실험은 제외하고, 최종 제출 파일이 만들어지는 과정만 포함합니다.

## 1. 발전 과정

초기에는 `stress_score`가 min 0, max 1, unique 101, spectrum이 0.01 단위로 관찰되는 점 바탕으로  
일정 범위의 선형 점수체계 가능성에 대한 가설을 세웠습니다.
- 가설1. 0~100 점수체계가 소수점화 되었고 각 변수별로 점수가 할당된 선형 가산식이 있을 것이다.
- 가설2. 0~1 점수체계가 소수 셋째자리에서 반올림되었고 각 변수별로 점수가 할당된 선형 가산식이 있을 것이다.(채택)

그러나 선형 모델에서 단순 가산식만으로는 target의 구조를 충분히 설명하기 어렵다고 판단하였고,  
변수 간 조합과 비선형 관계까지 반영되어 만들어졌을 가능성으로 관점을 넓혔습니다.
  
소수 둘째 자리로 반올림되었을 가능성은 유지한 채로,
2차·3차 feature, Tree/Boosting/신경망 계열과 비교한 결과 RBF-SVR이 가장 좋은 내부 검증 성능을 보여  
**최종 코드는 RBF-SVR 기준 비선형 관점에서 진행하였습니다.**

| 비교 항목 | 대표 접근 | CV MAE |
|---|---|---|
| 선형 모델 | Ridge, ElasticNet, LinearSVR | 0.244~0.250 |
| 다항·상호작용 | 2차·3차 feature, Polynomial SVR | 0.238 |
| Tree/Boosting | ExtraTrees, XGBoost, LightGBM, CatBoost | 0.180~0.200 |
| KNN/MLP | KNN, MLPRegressor | 0.218~0.246 |
| Kernel SVR | RBF-SVR | 0.13대 |

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

#누수 없이 train안에서만 fit하게 하려고 안전장치
from sklearn.base import BaseEstimator, TransformerMixin
#숫자형/범주형 전처리용
from sklearn.compose import ColumnTransformer
#결측값 안전 처리
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.svm import SVR


RANDOM_STATE = 42
TARGET = "stress_score"
ID_COL = "ID"

## 2. 데이터 로드


In [2]:
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "open"
SUBMISSIONS_DIR = PROJECT_ROOT / "submissions"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("project_root:", PROJECT_ROOT)
print("train:", train_df.shape)
print("test:", test_df.shape)
print("sample_submission:", sample_submission.shape)
train_df.head()


project_root: D:\python\stress
train: (3000, 18)
test: (3000, 17)
sample_submission: (3000, 2)


,ID,gender,age,height,weight,cholesterol,systolic_blood_pressure,diastolic_blood_pressure,glucose,bone_density,activity,smoke_status,medical_history,family_medical_history,sleep_pattern,edu_level,mean_working,stress_score
0,TRAIN_0000,F,72,161.49,58.47,279.84,165,100,143.35,0.87,moderate,ex-smoker,high blood pressure,diabetes,sleep difficulty,bachelors degree,NaN,0.63
1,TRAIN_0001,M,88,179.87,77.60,257.37,178,111,146.94,0.07,moderate,ex-smoker,NaN,diabetes,normal,graduate degree,NaN,0.83
2,TRAIN_0002,M,47,182.47,89.93,226.66,134,95,142.61,1.18,light,ex-smoker,NaN,NaN,normal,high school diploma,9.0,0.70
3,TRAIN_0003,M,69,185.78,68.63,206.74,158,92,137.26,0.48,intense,ex-smoker,high blood pressure,NaN,oversleeping,graduate degree,NaN,0.17
4,TRAIN_0004,F,81,164.63,71.53,255.92,171,116,129.37,0.34,moderate,ex-smoker,diabetes,diabetes,sleep difficulty,bachelors degree,NaN,0.36


## 3. feature engineering(최종 모델 기준)

1. `mean_working` 결측은 일반적인 평균 대체 대신, 최종 모델인 RBF-SVR의 거리 기반 표현에서  
결측군을 비결측 근무시간 범위와 분리하기 위해 sentinel 값 99로 처리했습니다.  
999도 유사한 결과였지만 설명 가능성과 과도한 값 사용을 피하기 위해 99를 선택했습니다.
2. 파생변수 `BMI`, `glucose/cholesterol`(두 대사지표의 상대적 비율), `cholesterol×glucose`(두 대사지표가 동시에 높을 때 정보)를 추가했습니다.
3. `gender`는 이진 변수로 map을 쓰고, 나머지 범주형 변수는 one-hot encoding으로 분리했습니다.
4. 결측률이 높은 범주형 변수들이 많아 결측 자체가 하나의 상태일 가능성이 있다고 판단했습니다.  
따라서 결측값을 추정해 채우기보다는 일반적으로 설문지에 "99.해당없음/미응답"이 있듯이 `"Unknown"`으로 처리했습니다.

In [3]:
# 각 fold에서 train에만 fit한 뒤 validation/test에는 transform만 적용되도록 하여 데이터 누수를 방지하려고 sklearn Pipeline 안에 넣기 위해 클래스로 구현했습니다. 

class DenseTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    #OneHotEncoder용, 0이 아주 많아질 것을 대비하여 dense array로 변환
    def transform(self, X):
        if hasattr(X, "toarray"):
            return X.toarray()
        return X


class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_df = X.copy()

        categorical_base = [
            "gender",
            "activity",
            "smoke_status",
            "medical_history",
            "family_medical_history",
            "sleep_pattern",
            "edu_level",
        ]
        for col in categorical_base:
            #결측값을 "Unknown"이라는 하나의 범주로 처리, one-hot 준비용
            X_df[f"{col}_cat"] = X_df[col].astype("object").fillna("Unknown")

        # mean_working 결측 -> sentinel 99
        X_df["mean_working"] = X_df["mean_working"].fillna(99.0)

        # 파생변수
        X_df["bmi"] = X_df["weight"] / np.square(X_df["height"] / 100.0)
        X_df["glucose_cholesterol_ratio"] = X_df["glucose"] / X_df["cholesterol"].replace(0, np.nan)
        X_df["cholesterol_glucose_product"] = X_df["cholesterol"] * X_df["glucose"]

        # 이진변수 gender
        X_df["gender_code"] = X_df["gender_cat"].map({"F": 0, "M": 1}).astype(float)

        # ID는 feature로 사용하지 않음
        return X_df.drop(columns=[ID_COL])


## 4. RBF-SVR pipeline

- 숫자형 변수: RobustScaler
- 범주형 변수(gender 제외): one-hot encoding
- model: RBF-SVR
- 후처리: 예측값을 0~1 범위로 제한하고 소수 둘째 자리로 반올림

RBF-SVR의 `C`와 `gamma`는 C/gamma grid search 결과에서 가장 좋았던 설정을 사용했습니다.
- C 후보: 1.0, 4.0, 8.0 (결과 차이 없음)
- gamma 후보: 0.5, 0.75, 1.0~1.2(미세조절), 1.5, 2.0  
- 최종 제출: C=1.0, gamma =1.06

In [7]:
NUMERIC_COLS = [
    "age",
    "height",
    "weight",
    "cholesterol",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "glucose",
    "bone_density",
    "mean_working",
    "bmi",
    "glucose_cholesterol_ratio",
    "cholesterol_glucose_product",
    "gender_code",
]

OHE_COLS = [
    "activity_cat",
    "sleep_pattern_cat",
    "edu_level_cat",
    "smoke_status_cat",
    "medical_history_cat",
    "family_medical_history_cat",
]


def build_model():
    #NUMERIC_COLS에는 robustscale, OHE_COLS에는 one-hot
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("scaler", RobustScaler()),
                    ]
                ),
                NUMERIC_COLS,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
                    ]
                ),
                OHE_COLS,
            ),
        ],
        # 명시한 컬럼만 남기기(안전장치)
        remainder="drop",
    )

    model = SVR(
        kernel="rbf",
        C=1.0,
        gamma=1.06,
        epsilon=0.0,
        shrinking=True,
        cache_size=500,
    )

    return Pipeline(
        steps=[
            ("features", FeatureEngineer()),
            ("preprocess", preprocessor),
            ("dense", DenseTransformer()),
            ("model", model),
        ]
    )


def postprocess(pred):
    return np.round(np.clip(np.asarray(pred, dtype=float), 0, 1), 2)

## 5. 10-fold CV

(submission는 **full train**에 다시 학습해서 생성하기 때문에 fold 수가 최종 prediction을 직접 바꾸지는 않습니다.)

fold 수는 여러 fold 후보를 비교하여 성능, fold 안정성, validation size의 균형이 좋은 10-fold KFold를 기준으로 사용했습니다.

| 검증 방식 | CV MAE | Fold std | Valid size |
|---|---:|---:|---:|
| 5-fold KFold | 0.149050 | 0.008868 | 600 |
| 10-fold KFold | 0.134170 | 0.007928 | 300 |
| 15-fold KFold | 0.131013 | 0.009900 | 200 |
| 20-fold KFold | 0.127323 | 0.011029 | 150 |
| 5-fold Stratified | 0.144327 | 0.005342 | - |
| 10-fold Stratified | 0.134607 | 0.011168 | - |

In [8]:
def run_cv(train, n_splits=10, random_state=RANDOM_STATE):
    X = train.drop(columns=[TARGET])
    y = train[TARGET].to_numpy()

    splitter = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    oof_pred = np.zeros(len(train), dtype=float)
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(splitter.split(X), start=1):
        model = build_model()
        model.fit(X.iloc[tr_idx], y[tr_idx])

        # 예측 + clip(0, 1) + round(2)
        # validation fold는 predict만 수행
        pred = postprocess(model.predict(X.iloc[va_idx]))
        oof_pred[va_idx] = pred
        
        mae = mean_absolute_error(y[va_idx], pred)

        fold_rows.append({"fold": fold, "valid_size": len(va_idx), "mae": mae})
        print(f"fold {fold:02d} | valid_size={len(va_idx):3d} | MAE={mae:.6f}")

    fold_df = pd.DataFrame(fold_rows)
    cv_mae = fold_df["mae"].mean()
    cv_std = fold_df["mae"].std(ddof=1)
    print("-" * 50)
    print(f"10-fold CV MAE: {cv_mae:.6f} ± {cv_std:.6f}")

    return fold_df, oof_pred


fold_df, oof_pred = run_cv(train_df, n_splits=10)
fold_df

fold 01 | valid_size=300 | MAE=0.128033
fold 02 | valid_size=300 | MAE=0.122267
fold 03 | valid_size=300 | MAE=0.136400
fold 04 | valid_size=300 | MAE=0.141933
fold 05 | valid_size=300 | MAE=0.139600
fold 06 | valid_size=300 | MAE=0.120100
fold 07 | valid_size=300 | MAE=0.137133
fold 08 | valid_size=300 | MAE=0.141233
fold 09 | valid_size=300 | MAE=0.140300
fold 10 | valid_size=300 | MAE=0.134633
--------------------------------------------------
10-fold CV MAE: 0.134163 ± 0.007948


,fold,valid_size,mae
0,1,300,0.128033
1,2,300,0.122267
2,3,300,0.136400
3,4,300,0.141933
4,5,300,0.139600
5,6,300,0.120100
6,7,300,0.137133
7,8,300,0.141233
8,9,300,0.140300
9,10,300,0.134633


## 6. Full train 학습 후 최종 제출 파일 생성

In [10]:
final_model = build_model()
final_model.fit(train_df.drop(columns=[TARGET]), train_df[TARGET].to_numpy())

test_pred = postprocess(final_model.predict(test_df))

submission = sample_submission.copy()
submission[TARGET] = test_pred

output_path = SUBMISSIONS_DIR / "submission_final_rbf_svr_lb012991.csv"
submission.to_csv(output_path, index=False)

print(f"saved: {output_path}")
print("prediction summary")
# 0~1 범위 잘 지켰는지 확인
print(pd.Series(test_pred).describe())
# clip(0, 1) 때문에 예측값이 너무 많이 0이나 1에 붙어버릴까봐 확인
print("endpoint counts:", {"pred_0": int(np.sum(test_pred == 0)), "pred_1": int(np.sum(test_pred == 1))})
submission.head()


saved: D:\python\stress\submissions\submission_final_rbf_svr_lb012991.csv
prediction summary
count    3000.000000
mean        0.496480
std         0.198116
min         0.000000
25%         0.470000
50%         0.490000
75%         0.520000
max         1.000000
dtype: float64
endpoint counts: {'pred_0': 4, 'pred_1': 5}


,ID,stress_score
0,TEST_0000,0.49
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.49
4,TEST_0004,0.53


## 7. 최종 결과

생성된 `submissions/submission_final_rbf_svr_lb012991.csv`가 최종 제출 파일입니다.

- 10-fold CV MAE: **0.134163**
- 최종 LB MAE: **0.129991**

